In [1]:
import matplotlib.pyplot as plt
import numpy as np
import sys, os, pickle
from pathlib import Path
import data_loading as dl
import scipy.io as sio


In [5]:
# US25, US33, US40, labels, batch_ids = dl.load_sensor_data()
# US25 = 25kHz (east)
# US33 = 33kHz (west)
# US40 = 40kHz (north)

RAW_DATA_PATH = "data_rot_chunks10.mat" 
ACTIONS = 21 # Total 21 actions (0-20) but we're skipping 6 - 9 & 21
SKIP_ACTIONS = [6, 7, 8, 9, 21]
CHUNKS = 10
ACTORS = 13

In [11]:
raw_data = sio.loadmat(RAW_DATA_PATH)
for key in raw_data.keys():
    print(type(raw_data[key]))
    print("\n")

<class 'bytes'>


<class 'str'>


<class 'list'>


<class 'numpy.ndarray'>


<class 'numpy.ndarray'>


<class 'numpy.ndarray'>


<class 'numpy.ndarray'>


<class 'numpy.ndarray'>


<class 'numpy.ndarray'>


<class 'numpy.ndarray'>


<class 'numpy.ndarray'>




In [ ]:
# def load_sensor_data():
"""
Load the sensor spectrogram data from the .mat file
Return US25_data, US33_data, US40_data, labels, batch_ids

Data is flattened into an ndarray containing ndarrays of spectrograms of action sequences.
"""

# Load raw data into individual sensor files
raw_data = sio.loadmat(RAW_DATA_PATH)
US25_data = raw_data['dataChunksUS25'][:, 0:ACTIONS, :]
US33_data = raw_data['dataChunksUS33'][:, 0:ACTIONS, :]
US40_data = raw_data['dataChunksUS40'][:, 0:ACTIONS, :]
rand_perm_idx = raw_data['randpermIx'] - 1

# Transpose the sensor data, create labels and batch ids. Empty sequences converted to None
US25_data_T = np.empty_like(US25_data)
US33_data_T = np.empty_like(US33_data)
US40_data_T = np.empty_like(US40_data)
labels = np.empty_like(US25_data)
batch_ids = np.empty_like(US25_data)
for chunk in range(CHUNKS):
    for action in range(ACTIONS):
        for actor in range(ACTORS):
            idx = (chunk, action, actor)
            # Check for empty sequences

            if US25_data[idx].size:
                US25_data_T[idx] = US25_data[idx].transpose()
                US33_data_T[idx] = US33_data[idx].transpose()
                US40_data_T[idx] = US40_data[idx].transpose()
                labels[idx] = action
                batch_ids[idx] = rand_perm_idx[idx]

# Flatten relo
US25 = US25_data_T.flatten()
US33 = US33_data_T.flatten()
US40 = US40_data_T.flatten()
labels = labels.flatten()
batch_ids = batch_ids.flatten()

# Find valid sequence ids
non_empty_ids = np.where(np.not_equal(labels, None))
# US25[non_empty_ids], US33[non_empty_ids], US40[non_empty_ids], \
#     labels[non_empty_ids], batch_ids[non_empty_ids]

In [8]:
print(US40[non_empty_ids].shape)

(2700,)
